# Active Falsification — pilot on Colab

Runs the small-model pilot end to end on a single Colab GPU.

Order (do not skip steps):
1. install requirements
2. check the GPU
3. run the mathematical self-tests — **stop here if they fail**
4. preview generated problems by hand
5. run the **first** small model
6. display the calibration summary and the main table
7. optionally run the second small model

**4-bit pilot numbers must never be pooled with full-precision results
without a visible warning.** The run manifest records this automatically.


## 1. Get the code and install requirements

In [ ]:
# If you are running from a clone, set REPO to its path instead.
import os, sys
REPO = "/content/active-falsification"
if not os.path.exists(REPO):
    # replace with your own clone URL, or upload the directory to /content
    !git clone https://example.invalid/active-falsification.git {REPO} || echo "clone the repo or upload it to /content/active-falsification"
os.chdir(REPO)
print(os.getcwd())

In [ ]:
!pip -q install -r requirements.txt
!pip -q install bitsandbytes

## 2. Check the GPU

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())
    free, total = torch.cuda.mem_get_info()
    print(f"memory: {free/2**30:.1f} GiB free / {total/2**30:.1f} GiB total")
else:
    print("No GPU. Runtime > Change runtime type > GPU (T4 is enough for the 4-bit pilot).")

## 3. Mathematical self-tests — mandatory

Do not run a model until this prints `SELF TEST PASSED`.

In [ ]:
!python scripts/self_test.py

## 4. Inspect generated problems by hand

Read these. This is the cheapest place to catch a broken generator.

In [ ]:
!python scripts/generate_preview.py --n 3
!python scripts/generate_preview.py --n 2 --challenge

In [ ]:
# how much of the hidden label does each queryable feature explain?
!python scripts/audit_schema.py --n 400

## 5. Offline plumbing check (no GPU, seconds)

Exercises the whole pipeline with the deterministic mock model.

In [ ]:
!python scripts/run_mock.py --skip-self-test

## 6. First small model

One model at a time — models are loaded and unloaded serially to keep GPU memory flat.

In [ ]:
!python scripts/run_experiment.py --config configs/pilot.yaml --models smollm3_3b --tag smollm3

## 7. Calibration summary and main table

In [ ]:
import glob, os
import pandas as pd

runs = sorted(glob.glob("outputs/*_pilot_smollm3"))
RUN = runs[-1]
print("run:", RUN)

print("\n--- calibration summary (pooled rows drive the difficulty choice) ---")
display(pd.read_csv(os.path.join(RUN, "calibration_summary.csv")))

import json
print("\n--- selected difficulties ---")
print(json.dumps(json.load(open(os.path.join(RUN, "selected_difficulties.json"))), indent=2))

In [ ]:
print("--- main table ---")
display(pd.read_csv(os.path.join(RUN, "main_table.csv")))

print("--- token accounting (Active vs Passive budget) ---")
!python scripts/make_tables.py --run-dir {RUN}
display(pd.read_csv(os.path.join(RUN, "token_accounting.csv")))

### Read some raw outputs

Never report a number you have not traced back to a raw response.

In [ ]:
import json
rows = [json.loads(l) for l in open(os.path.join(RUN, "evaluation_raw.jsonl"))]
for r in rows[:3]:
    print("=" * 70)
    print(r["family"], "|", r["condition"], "|", r["split"],
          "| gold:", r["gold_strategy_applies"], "| predicted:", r["predicted"])
    print(r["raw_response"][:600])

## 8. Optional: second small model

Run this only after you have looked at the first model's output.

In [ ]:
!python scripts/run_experiment.py --config configs/pilot.yaml --models phi35_mini --tag phi35 --skip-self-test

## 9. Before you use any of this

Work through `docs/AUTHOR_REVIEW_CHECKLIST.md`. In particular:

- these are **4-bit pilot** numbers;
- `phi35_mini` is ~3.8B, not an exact 3B model;
- decide which `active_schema_profile` the paper reports, and say so;
- report the realised `A_tokens` / `P_tokens` numbers, not the phrase "compute matched".